# 02 — Hawkes fit and goodness-of-fit on real trade arrivals

Fit the 2-dim (buy/sell aggressor) exponential Hawkes to the fixture hour and test it with Ogata's time-rescaling residuals. The Rust CLI (`quant-sim fit hawkes`) fits the same model; a slow test holds the two implementations to matching likelihoods.

In [ ]:
FIXTURE_ONLY = True  # papermill parameter

In [ ]:
import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from quantsim_research.config import repo_root

FIXTURE = repo_root() / "data" / "fixtures" / "binance-um" / "trades" / "BTCUSDT-2024-01-07-h00.csv"
ARTIFACTS = repo_root() / "artifacts" / "notebooks"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
df = pl.read_csv(FIXTURE)
ts_ns = df["ts_ns"].to_numpy()
price = df["price_e8"].to_numpy() / 1e8
side = df["side"].to_numpy()
print(f"fixture: {len(df)} trades, {(ts_ns[-1]-ts_ns[0])/1e9:.0f}s span")


In [ ]:
from quantsim_research.hawkes.exp_mle import fit_exp_hawkes

seconds = (ts_ns - ts_ns[0]) / 1e9
events = [seconds[side == 1], seconds[side == -1]]
t_end = float(seconds[-1])
if FIXTURE_ONLY:
    # Subsample to the first 10 minutes so the notebook executes in seconds.
    events = [e[e < 600.0] for e in events]
    t_end = 600.0
fit = fit_exp_hawkes(events, t_end)
print(f"mu={fit.mu.round(3)}  beta={fit.beta:.1f}/s  rho(G)={fit.spectral_radius:.3f}")
print(f"logL={fit.log_likelihood:.1f} vs Poisson {fit.poisson_log_likelihood:.1f} "
      f"(delta={fit.log_likelihood - fit.poisson_log_likelihood:.0f})")


## Time-rescaling residuals
Under the fitted model, compensator-transformed inter-arrivals should be iid Exp(1): KS test + QQ plot.

In [ ]:
from scipy import stats

from quantsim_research.hawkes.diagnostics import time_rescaling_residuals

res = time_rescaling_residuals(events, fit.mu, fit.alpha, fit.beta, dim=0)
print(f"KS={res.ks_statistic:.4f} p={res.ks_pvalue:.3f}  mean tau={res.residuals.mean():.3f}")
theo = stats.expon.ppf((np.arange(1, res.residuals.size + 1) - 0.5) / res.residuals.size)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(theo, np.sort(res.residuals), '.', ms=2)
lim = max(theo.max(), res.residuals.max())
ax.plot([0, lim], [0, lim], 'r--', lw=1)
ax.set_xlabel("Exp(1) quantiles"); ax.set_ylabel("rescaled residuals"); ax.set_title("QQ")
fig.tight_layout(); fig.savefig(ARTIFACTS / "02_qq.png", dpi=120)
